# **Video Processing Week 1: Fondasi & Teknik CV Klasik**

Minggu ini kita akan mempelajari **pengolahan video frame-by-frame** dengan teknik-teknik klasik (non-AI) menggunakan OpenCV. Fokus utama adalah manipulasi video real-time maupun file, serta efek dasar yang sering digunakan dalam produksi video.

**Tujuan Pembelajaran:**

Di akhir sesi ini kita akan bisa:
- Membaca, menulis, dan menangkap video (file & webcam).
- Mengkonversi colorspace dan menerapkan filter dasar (blur, edge).
- Membuat efek *chroma key* (green screen) dengan mask warna.
- Melakukan *object tracking* menggunakan algoritma OpenCV (CSRT/KCF).
- (Opsional) Men-stabilkan video dengan fitur bawaan OpenCV.

> *This module is inspired by the development of last semester’s materials.* 

## **Identitas Mahasiswa**

- **Nama:** Cidesta Mentari Marintan Simbolon  
- **NIM:** 122400096  
- **Kelas:** Pengolahan Citra

### **Membaca file video**

Kita akan mencoba membuka file video menggunakan OpenCV dan menampilkan:
- frame pertama dan frame ke-n
- Video di dalam window terpisah

In [1]:
import cv2
import os

# path video
PATH_VIDEO = os.path.join(os.getcwd(), "data", "man_walking.mp4")

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    print("Error: tidak bisa membuka video")
    exit()

while True:
    ret, frame = cap.read()

    if not ret:
        print("Video selesai atau gagal membaca frame")
        break

    # tampilkan video
    cv2.imshow("Video Full Playback", frame)

    # tekan q untuk keluar
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Video selesai atau gagal membaca frame


>>Pada bagian ini, saya mengubah metode yang sebelumnya hanya menampilkan satu frame dari video menjadi menampilkan video secara penuh (full playback). Jika pada kode awal video hanya diambil satu frame saja untuk ditampilkan, pada versi yang saya buat video dibaca menggunakan loop sehingga setiap frame dapat ditampilkan secara berurutan seperti pemutar video. Dengan perubahan ini, proses visualisasi menjadi lebih interaktif karena seluruh isi video dapat diamati secara real-time. Selain itu, saya juga menambahkan kontrol tombol ‘q’ untuk menghentikan video agar program lebih fleksibel saat dijalankan.

In [2]:
import cv2
import os

PATH_VIDEO = os.path.join(os.getcwd(), "data", "man_walking.mp4")

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    print("Error: video tidak dapat dibuka")
else:
    print("Video berhasil diputar. Tekan 'q' untuk keluar.")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Video selesai diputar")
            break

        # resize (modifikasi biar lebih stabil di layar kecil)
        frame = cv2.resize(frame, (800, 450))

        cv2.imshow("Custom Video Player", frame)

        # kontrol keluar
        if cv2.waitKey(25) & 0xFF == ord("q"):
            print("Video dihentikan oleh user")
            break

cap.release()
cv2.destroyAllWindows()

Video berhasil diputar. Tekan 'q' untuk keluar.
Video selesai diputar


>>Pada bagian ini, saya melakukan modifikasi dari kode dosen yang sebelumnya hanya menampilkan video secara langsung menggunakan OpenCV. Pada versi yang saya buat, saya menambahkan beberapa perubahan seperti pengecekan status pembukaan video yang lebih jelas serta pesan informasi saat video berhasil diputar atau selesai. Selain itu, saya juga menambahkan fitur resize frame agar ukuran video lebih konsisten dan nyaman ditampilkan di layar. Dengan perubahan ini, proses playback video menjadi lebih rapi, informatif, dan lebih mudah dipahami saat dijalankan dibandingkan versi sebelumnya.

### Menangkap Gambar dari Webcam

Untuk menangkap video dari webcam, kita bisa menggunakan `cv2.VideoCapture(0)`, di mana `0` adalah indeks default untuk webcam pertama. Jika Anda memiliki beberapa kamera, Anda bisa mencoba `1`, `2`, dll.

In [4]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("Webcam aktif. Tekan 'q' untuk keluar.")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Error: frame tidak terbaca")
            break

        # MODIF: flip biar seperti mirror (lebih natural)
        frame = cv2.flip(frame, 1)

        # MODIF: resize biar stabil di semua layar
        frame = cv2.resize(frame, (800, 450))

        # MODIF: tambah teks status
        cv2.putText(frame, "LIVE CAMERA", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Custom Webcam View", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Webcam dihentikan oleh user")
            break

cap.release()
cv2.destroyAllWindows()

Webcam aktif. Tekan 'q' untuk keluar.
Webcam dihentikan oleh user


>>Pada bagian ini, saya melakukan pengembangan dari kode dosen yang sebelumnya hanya menampilkan webcam secara langsung menggunakan OpenCV. Pada versi yang saya buat, saya menambahkan beberapa fitur tambahan seperti flip horizontal agar tampilan webcam terlihat seperti cermin sehingga lebih natural, serta resize frame agar ukuran tampilan lebih konsisten di berbagai ukuran layar. Selain itu, saya juga menambahkan teks “LIVE CAMERA” sebagai indikator bahwa program sedang berjalan secara real-time. Dengan modifikasi ini, tampilan webcam menjadi lebih interaktif dan lebih informatif dibandingkan versi awal.

**### Menyimpan/menulis ke file video baru**

Untuk menyimpan video yang telah diproses, kita dapat menggunakan `cv2.VideoWriter`. Kita perlu menentukan nama file output, codec video, frame rate, dan ukuran frame.

Kita akan mencoba mengubah video menjadi grayscale dan menyimpannya ke file baru.

In [6]:
import cv2
import os

PATH_VIDEO = os.path.join(os.getcwd(), "data", "man_walking.mp4")
OUTPUT_VIDEO = os.path.join(os.getcwd(), "data", "output_gray_custom.mp4")

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    print("Error: video tidak bisa dibuka")
else:
    # ambil info video
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

    print("Processing video grayscale dimulai...")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Video selesai diproses")
            break

        # MODIF 1: grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # MODIF 2: tambah efek (biar beda dari dosen)
        gray = cv2.equalizeHist(gray)

        # convert lagi ke BGR untuk disimpan
        gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

        # tulis ke file
        out.write(gray_bgr)

        # tampilkan preview
        preview = cv2.resize(gray, (800, 450))
        cv2.imshow("Custom Grayscale Processing", preview)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("Dihentikan oleh user")
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print("Output tersimpan di:", OUTPUT_VIDEO)

Processing video grayscale dimulai...
Video selesai diproses
Output tersimpan di: c:\Users\USER\Documents\TUGAS AKU\PENGOLAHAN CITRA\Pengolahan_Multimedia\data\output_gray_custom.mp4


>>Pada bagian ini, saya melakukan pengembangan dari kode dosen yang sebelumnya hanya mengubah video menjadi grayscale dan menyimpannya langsung. Pada versi yang saya buat, saya menambahkan peningkatan berupa histogram equalization untuk meningkatkan kontras pada video grayscale sehingga hasil tampilan menjadi lebih jelas. Selain itu, saya juga menambahkan preview display yang di-resize agar lebih nyaman dilihat saat proses berjalan. Dengan modifikasi ini, hasil video tidak hanya menjadi grayscale, tetapi juga memiliki kualitas visual yang lebih baik dibandingkan versi awal.

## Manipulasi Frame Dasar

Setelah berhasil membaca dan menulis video, kita akan mempelajari teknik-teknik dasar untuk memanipulasi setiap frame dalam video. Manipulasi frame adalah fondasi dari semua efek video processing:

### **Konversi Colorspace**
Mengubah representasi warna frame (BGR → Grayscale, BGR → HSV, dll) untuk kebutuhan analisis atau efek tertentu.

**1. BGR to Grayscale:**
- Mengubah gambar berwarna menjadi hitam-putih
- Menghilangkan informasi warna, hanya menyimpan intensitas cahaya
- Mengurangi 3 channel (Blue, Green, Red) menjadi 1 channel saja
- Berguna untuk analisis bentuk, deteksi tepi, dan mengurangi kompleksitas komputasi

**2. BGR to HSV:**
- **H (Hue)**: Jenis warna (0-179° dalam OpenCV)
- **S (Saturation)**: Tingkat kejenuhan/kemurnian warna (0-255)
- **V (Value)**: Kecerahan/intensitas warna (0-255)
- Lebih intuitif untuk pemrosesan berdasarkan warna tertentu
- Sangat berguna untuk chroma keying (green screen) dan color-based object detection

In [2]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("Webcam grayscale kuat aktif. Tekan 'q' atau 'y' untuk keluar.")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        # mirror biar natural
        frame = cv2.flip(frame, 1)

        # grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 🔥 MODIF UTAMA: perkuat efek grey (kontras lebih tegas)
        gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=-30)

        # resize
        gray = cv2.resize(gray, (800, 450))

        cv2.putText(gray, "DEEP GRAYSCALE MODE", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, 255, 2)

        cv2.imshow("Strong Grayscale Webcam", gray)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q') or key == ord('y'):
            break

cap.release()
cv2.destroyAllWindows()

Webcam grayscale kuat aktif. Tekan 'q' atau 'y' untuk keluar.


>>Pada bagian ini, saya melakukan pengembangan dari kode grayscale sebelumnya dengan meningkatkan intensitas efek abu-abu agar hasil tampilan lebih tegas dan tidak terlalu terang. Hal ini dilakukan dengan menambahkan pengaturan kontras dan brightness menggunakan `convertScaleAbs` sehingga warna pada frame menjadi lebih “deep grayscale”. Selain itu, tetap dipertahankan fitur mirror dan resize agar tampilan lebih nyaman dilihat. Dengan modifikasi ini, hasil grayscale menjadi lebih kuat dan lebih jelas dibandingkan versi sebelumnya.

In [3]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

# CLAHE untuk kontras lokal (biar grey lebih hidup)
clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("Webcam cinematic grayscale aktif. Tekan q / y / x untuk keluar.")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Frame tidak terbaca")
            break

        # mirror effect
        frame = cv2.flip(frame, 1)

        # grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 🔥 upgrade 1: CLAHE (kontras lokal lebih realistis)
        gray = clahe.apply(gray)

        # 🔥 upgrade 2: noise ringan biar cinematic / CCTV feel
        noise = np.random.normal(0, 3, gray.shape).astype(np.uint8)
        gray = cv2.add(gray, noise)

        # resize preview
        gray = cv2.resize(gray, (800, 450))

        # UI text
        cv2.putText(gray, "CINEMATIC GRAYSCALE MODE", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)

        cv2.imshow("Custom Cinematic Webcam", gray)

        key = cv2.waitKey(1) & 0xFF

        # 🔥 MODIF EXIT MULTI KEY
        if key == ord('q') or key == ord('y') or key == ord('x'):
            print("Webcam dihentikan")
            break

cap.release()
cv2.destroyAllWindows()

Webcam cinematic grayscale aktif. Tekan q / y / x untuk keluar.
Webcam dihentikan


>>Pada bagian ini, saya melakukan pengembangan dari kode grayscale webcam sebelumnya dengan menambahkan efek visual yang lebih realistis dan cinematic. Saya menggunakan CLAHE (Contrast Limited Adaptive Histogram Equalization) untuk meningkatkan kontras lokal sehingga hasil grayscale menjadi lebih detail tanpa menghilangkan karakter abu-abu. Selain itu, saya juga menambahkan sedikit noise pada frame agar tampilan terlihat seperti CCTV atau video monitoring yang lebih natural. Tidak hanya itu, saya juga memodifikasi kontrol keluar program dengan menambahkan beberapa tombol yaitu q, y, dan x agar lebih fleksibel dibandingkan versi sebelumnya.

In [4]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("AI Grayscale Mode aktif. Tekan q / y / x untuk keluar.")

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            print("Frame tidak terbaca")
            break

        # mirror effect
        frame = cv2.flip(frame, 1)

        # grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 🔥 upgrade 1: enhance contrast
        gray = cv2.convertScaleAbs(gray, alpha=1.5, beta=-25)

        # 🔥 upgrade 2: edge detection (biar lebih “AI vision”)
        edges = cv2.Canny(gray, 60, 120)

        # gabungkan grayscale + edge
        combined = cv2.addWeighted(gray, 0.8, edges, 0.2, 0)

        # resize
        combined = cv2.resize(combined, (800, 450))

        # UI text
        cv2.putText(combined, "AI GRAYSCALE EDGE MODE", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)

        cv2.imshow("Custom AI Grayscale", combined)

        key = cv2.waitKey(1) & 0xFF

        # 🔥 multi exit key (biar beda dari dosen)
        if key == ord('q') or key == ord('y') or key == ord('x'):
            print("Webcam dihentikan")
            break

cap.release()
cv2.destroyAllWindows()

AI Grayscale Mode aktif. Tekan q / y / x untuk keluar.
Webcam dihentikan


>>Pada bagian ini, saya melakukan pengembangan dari kode grayscale webcam sebelumnya dengan menambahkan teknik pemrosesan citra yang lebih lanjut. Selain meningkatkan kontras menggunakan `convertScaleAbs`, saya juga menambahkan deteksi tepi (Canny Edge Detection) untuk menonjolkan struktur objek pada frame. Hasil grayscale kemudian digabungkan dengan edge detection sehingga menghasilkan tampilan yang lebih informatif seperti sistem visual berbasis AI. Selain itu, saya juga menambahkan beberapa tombol keluar (q, y, x) agar kontrol program lebih fleksibel dibandingkan versi sebelumnya.

In [23]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("Webcam HSV Mode (q/y/x untuk keluar)")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: Gagal membaca frame")
            break

        # mirror biar enak dilihat
        frame = cv2.flip(frame, 1)

        # konversi ke HSV (ini inti kamu)
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # resize biar lebih enak
        hsv_frame = cv2.resize(hsv_frame, (800, 450))

        # teks
        cv2.putText(hsv_frame, "HSV RAW VIEW MODE", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        cv2.imshow("HSV Live View", hsv_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == ord('y') or key == ord('x'):
            break

cap.release()
cv2.destroyAllWindows()

Webcam HSV Mode (q/y/x untuk keluar)


>>Pada bagian ini, saya menampilkan hasil konversi webcam ke ruang warna HSV secara real-time. Berbeda dengan format BGR yang biasanya digunakan untuk tampilan visual, HSV memisahkan komponen warna menjadi hue, saturation, dan value sehingga hasil tampilannya terlihat berbeda atau tidak natural. Program ini hanya melakukan konversi tanpa proses tambahan agar dapat melihat langsung representasi warna dalam ruang HSV.

### **Filter Dasar**

Setelah konversi colorspace, kita akan mempelajari filter dasar yang sering digunakan dalam video processing:

**1. Blur Filter:**
- Menghaluskan gambar dengan mengurangi noise dan detail halus
- Membuat efek buram dengan merata-ratakan nilai pixel di sekitarnya
- Berguna untuk preprocessing sebelum deteksi objek atau mengurangi gangguan
- Jenis blur: Gaussian Blur, Motion Blur, dan Median Blur

**2. Canny Edge Detection:**
- Mendeteksi tepi/garis batas objek dalam gambar
- Mengubah gambar menjadi representasi garis putih pada latar belakang hitam
- Sangat berguna untuk analisis bentuk, kontur objek, dan segmentasi
- Menggunakan algoritma gradien untuk menemukan perubahan intensitas yang tajam

In [24]:
import cv2

cap = cv2.VideoCapture(0)

mode = 0  # 0 = normal, 1 = blur

if not cap.isOpened():
    print("Webcam tidak bisa diakses")
else:
    print("Tekan q/y/x = keluar | spasi = ganti mode")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        key = cv2.waitKey(1) & 0xFF

        # ganti mode blur / normal
        if key == 32:  # SPACE
            mode = 1 - mode

        # keluar
        if key in [ord('q'), ord('y'), ord('x')]:
            break

        if mode == 1:
            output = cv2.blur(frame, (21, 21))  # blur lebih kuat
            text = "BLUR MODE"
        else:
            output = frame
            text = "NORMAL MODE"

        cv2.putText(output, text, (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        cv2.imshow("Blur Webcam (Custom)", output)

cap.release()
cv2.destroyAllWindows()

Tekan q/y/x = keluar | spasi = ganti mode


>>Pada bagian ini, saya mengembangkan program blur sederhana dengan menambahkan fitur mode switching agar pengguna dapat berpindah antara tampilan normal dan blur secara real-time. Efek blur menggunakan average blur dengan kernel yang diperbesar untuk menghasilkan efek yang lebih halus. Selain itu, saya menambahkan kontrol tombol agar program lebih interaktif, yaitu tombol spasi untuk mengganti mode serta beberapa tombol keluar yang lebih fleksibel.

In [5]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam tidak bisa diakses")
else:
    print("Webcam Canny Edge aktif. Tekan 'q' / 'y' / 'x' untuk keluar.")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: Gagal membaca frame")
            break

        # mirror biar natural
        frame = cv2.flip(frame, 1)

        # grayscale (wajib untuk Canny)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 🔥 minor upgrade: sedikit smoothing biar edge lebih stabil
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        # Canny Edge Detection (tetap sama konsep dosen)
        edges = cv2.Canny(gray, 100, 150)

        # 🔥 minor upgrade: bikin edge lebih “tegas” sedikit
        edges = cv2.dilate(edges, None, iterations=1)

        cv2.imshow("Canny Edge Detection", edges)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == ord('y') or key == ord('x'):
            print("Program dihentikan")
            break

cap.release()
cv2.destroyAllWindows()

Webcam Canny Edge aktif. Tekan 'q' / 'y' / 'x' untuk keluar.
Program dihentikan


>>Pada bagian ini, saya tetap menggunakan metode Canny Edge Detection seperti pada kode sebelumnya, namun dengan sedikit pengembangan untuk meningkatkan kualitas hasil deteksi. Saya menambahkan Gaussian Blur sebelum proses Canny agar noise pada gambar berkurang sehingga hasil edge lebih stabil. Selain itu, saya juga menambahkan proses dilasi ringan untuk mempertegas garis tepi yang dihasilkan. Secara konsep tetap sama dengan versi dasar, hanya saja hasil visualnya menjadi lebih jelas.

**## Chroma Keying (Efek "Green Screen")**

Chroma keying adalah teknik mengganti background berdasarkan warna tertentu (biasanya hijau atau biru) dengan gambar/video lain. Teknik ini sangat populer dalam industri film dan broadcasting.

**Konsep Dasar:**
- **Background khusus**: Objek direkam di depan layar berwarna solid (green screen/blue screen)
- **Color separation**: Algoritma memisahkan warna background dari foreground
- **Compositing**: Background diganti dengan gambar/video yang diinginkan

**Catatan**:
- Hindari objek yang berwarna hijau agar tidak ikut ter-mask
- Kualitas hasil tergantung pada kualitas green screen dan pencahayaan

**Langkah-langkah Chroma Keying:**

### **1. Membuat Mask Berdasarkan Rentang Warna (Hijau)**
- Konversi frame ke HSV colorspace untuk deteksi warna yang lebih akurat
- Tentukan rentang nilai HSV untuk warna hijau (Hue: 40-80, Saturation: tinggi, Value: tinggi)
- Gunakan `cv2.inRange()` untuk membuat binary mask (putih = hijau, hitam = bukan hijau)
- Terapkan **morphological operations** untuk membersihkan mask:
    - **Erosi**: menghilangkan titik-titik noise kecil
    - **Dilasi**: menutup lubang-lubang kecil di dalam mask
    - Hasil: mask yang lebih bersih dan solid

In [6]:
import cv2
import numpy as np
import os

PATH_VIDEO = os.path.join(os.getcwd(), 'data', 'green_screen_sample.mp4')

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    print("Error: video tidak bisa dibuka")
else:
    print("Green Screen Detection aktif (tekan q untuk keluar)")

    # kernel morfologi (dipakai sekali biar efisien)
    kernel = np.ones((5, 5), np.uint8)

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Video selesai")
            break

        # mirror biar lebih natural (minor modifikasi dari dosen)
        frame = cv2.flip(frame, 1)

        # HSV conversion
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        # range hijau
        lower_green = np.array([40, 40, 40])
        upper_green = np.array([80, 255, 255])

        # mask
        mask = cv2.inRange(hsv, lower_green, upper_green)

        # 🔥 sedikit improvement cleaning
        mask = cv2.GaussianBlur(mask, (3, 3), 0)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        # hasil segmentasi
        result = cv2.bitwise_and(frame, frame, mask=mask)

        # resize biar rapi
        frame = cv2.resize(frame, (400, 300))
        mask = cv2.resize(mask, (400, 300))
        result = cv2.resize(result, (400, 300))

        # ubah mask jadi BGR biar bisa digabung
        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)

        # gabung 3 tampilan
        output = np.hstack((frame, mask_bgr, result))

        cv2.imshow("GREEN SCREEN ANALYSIS (MODIFIED)", output)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

Green Screen Detection aktif (tekan q untuk keluar)


>>Pada bagian ini, saya melakukan pengembangan dari metode green screen segmentation dengan menggunakan ruang warna HSV. Proses dilakukan dengan menentukan rentang warna hijau kemudian membuat binary mask menggunakan fungsi inRange. Setelah itu, saya menambahkan proses pembersihan noise menggunakan Gaussian Blur serta operasi morfologi opening dan closing agar hasil mask lebih stabil dan tidak berisik. Hasil akhirnya ditampilkan dalam bentuk segmentasi objek sehingga objek selain warna hijau dapat dipisahkan dari background secara lebih jelas.

### **2. Mengganti Area Mask dengan Gambar atau Video Lain**
- Load 2 video: Green screen dan background replacement
- Deteksi hijau: Konversi ke HSV, buat mask untuk area hijau dengan cv2.inRange()
- Bersihkan mask: Morphological operations (OPEN & CLOSE) untuk hapus noise
- Pisahkan objek & background: bitwise_and() dengan mask asli dan mask terbalik
- Gabungkan hasil: cv2.add() foreground + background = green screen diganti

In [7]:
import cv2
import numpy as np
import os

PATH_VIDEO_GREEN = os.path.join(os.getcwd(), 'data', 'green_screen_sample.mp4')
PATH_VIDEO_BG = os.path.join(os.getcwd(), 'data', 'man_walking.mp4')

cap_green = cv2.VideoCapture(PATH_VIDEO_GREEN)
cap_bg = cv2.VideoCapture(PATH_VIDEO_BG)

if not cap_green.isOpened() or not cap_bg.isOpened():
    print("Error membuka video")
else:
    print("Chroma Key Advanced aktif (q untuk keluar)")

    kernel = np.ones((5, 5), np.uint8)

    while True:
        ret_g, fg = cap_green.read()
        ret_b, bg = cap_bg.read()

        if not ret_g or not ret_b:
            break

        # samakan ukuran
        bg = cv2.resize(bg, (fg.shape[1], fg.shape[0]))

        fg = cv2.flip(fg, 1)

        hsv = cv2.cvtColor(fg, cv2.COLOR_BGR2HSV)

        lower_green = np.array([40, 40, 40])
        upper_green = np.array([80, 255, 255])

        mask = cv2.inRange(hsv, lower_green, upper_green)

        # 🔥 upgrade 1: smoothing mask biar tidak kasar
        mask = cv2.GaussianBlur(mask, (7, 7), 0)

        # 🔥 upgrade 2: cleaning tetap
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # invert
        mask_inv = cv2.bitwise_not(mask)

        # foreground & background
        fg_part = cv2.bitwise_and(fg, fg, mask=mask_inv)
        bg_part = cv2.bitwise_and(bg, bg, mask=mask)

        # 🔥 upgrade 3: blend halus (biar tidak “cut and paste”)
        result = cv2.addWeighted(fg_part, 1, bg_part, 1, 0)

        # 🔥 upgrade 4: sedikit brightness matching
        result = cv2.convertScaleAbs(result, alpha=1.05, beta=5)

        result = cv2.resize(result, (800, 450))

        cv2.putText(result, "CHROMA KEY ADVANCED (MODIFIED)", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

        cv2.imshow("Chroma Key Advanced", result)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

cap_green.release()
cap_bg.release()
cv2.destroyAllWindows()

Chroma Key Advanced aktif (q untuk keluar)


>>Pada bagian ini, saya melakukan pengembangan dari metode chroma key sebelumnya dengan menambahkan beberapa teknik peningkatan kualitas visual. Saya menambahkan Gaussian Blur pada mask untuk menghaluskan transisi antara foreground dan background sehingga hasil tidak terlihat kasar. Selain itu, saya juga menggunakan blending tambahan untuk menggabungkan kedua layer secara lebih natural. Saya juga menambahkan penyesuaian brightness ringan agar hasil akhir terlihat lebih seimbang antara video foreground dan background. Dengan perubahan ini, hasil chroma key menjadi lebih halus dan realistis dibandingkan versi sebelumnya.

**### Praktik: Memilih objek (via ROI) dan membiarkan tracker mengikutinya**

In [2]:
# Optical Flow Tracking (FIXED + lebih stabil)
import os
import cv2
import numpy as np

PATH_VIDEO = os.path.join(os.getcwd(), 'data', 'man_walking.mp4')

cap = cv2.VideoCapture(PATH_VIDEO)

lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

if cap.isOpened():
    ret, old_frame = cap.read()
    if not ret:
        print("Video tidak bisa dibaca")
        exit()

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

    # 🔥 FIX: ambil titik otomatis (ini yang bikin GERAK)
    p0 = cv2.goodFeaturesToTrack(
        old_gray,
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    mask = np.zeros_like(old_frame)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optical Flow
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel()
                c, d = old.ravel()

                frame = cv2.line(frame, (int(a), int(b)), (int(c), int(d)), (0, 255, 0), 2)
                frame = cv2.circle(frame, (int(a), int(b)), 3, (0, 0, 255), -1)

            p0 = good_new.reshape(-1, 1, 2)

        old_gray = frame_gray.copy()

        cv2.imshow("Optical Flow FIXED (GERAK)", frame)

        if cv2.waitKey(20) & 0xFF == ord('y'):
            break

cap.release()
cv2.destroyAllWindows()

>>Pada percobaan ini, saya melakukan object tracking pada video menggunakan metode Optical Flow Lucas-Kanade. Berbeda dengan metode tracking sebelumnya yang menggunakan bounding box seperti CSRT, pada metode ini saya tidak lagi menandai objek secara langsung, tetapi menggunakan titik-titik fitur yang dideteksi otomatis dari frame pertama menggunakan Shi-Tomasi Corner Detection. Titik-titik tersebut kemudian saya lacak pergerakannya pada frame berikutnya sehingga terlihat arah dan pola gerakan objek secara real-time. Hasilnya, objek dapat diikuti berdasarkan perpindahan titik antar frame tanpa perlu menggunakan modul tracking khusus dari OpenCV, sehingga lebih ringan dan tetap bisa berjalan stabil.

### 🔎 Eksplorasi

- Cobalah melakukan Object Detection melalui Webcam Anda secara real-time
- Cobalah menggunakan algoritma tracker lainnya dan bandingkan hasilnya.

Hint: ubah bagian kode `tracker = cv2.TrackerCSRT_create()`

In [9]:
### 🔎 Mencoba Eksplorasi

import cv2

# =========================
# PILIH TRACKER DI SINI
# =========================
TRACKER_TYPE = "CSRT"   # ganti: "CSRT", "KCF", "MOSSE", "MIL"

# =========================
# CREATE TRACKER (ANTI ERROR OPENCV BARU)
# =========================
def create_tracker(tracker_type):
    if tracker_type == "CSRT":
        return cv2.legacy.TrackerCSRT_create() if hasattr(cv2, 'legacy') else cv2.TrackerCSRT_create()
    elif tracker_type == "KCF":
        return cv2.legacy.TrackerKCF_create() if hasattr(cv2, 'legacy') else cv2.TrackerKCF_create()
    elif tracker_type == "MOSSE":
        return cv2.legacy.TrackerMOSSE_create() if hasattr(cv2, 'legacy') else cv2.TrackerMOSSE_create()
    elif tracker_type == "MIL":
        return cv2.legacy.TrackerMIL_create() if hasattr(cv2, 'legacy') else cv2.TrackerMIL_create()
    else:
        raise ValueError("Tracker tidak dikenali")

# =========================
# WEBCAM
# =========================
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    print("Webcam tidak bisa dibuka")
    exit()

# pilih objek manual
bbox = cv2.selectROI("Pilih objek", frame, False)
cv2.destroyWindow("Pilih objek")

# init tracker
tracker = create_tracker(TRACKER_TYPE)
tracker.init(frame, bbox)

# =========================
# LOOP TRACKING
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, f"{TRACKER_TYPE} Tracking",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (0, 255, 0), 2)
    else:
        cv2.putText(frame, "Lost Tracking",
                    (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 0, 255), 2)

    cv2.imshow("Object Tracking", frame)

    # ESC untuk keluar
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

>> Program ini melakukan object tracking secara real-time menggunakan webcam dengan cara memilih terlebih dahulu objek yang ingin dilacak melalui bounding box, lalu sistem akan mengikuti pergerakan objek tersebut pada setiap frame video menggunakan algoritma tracker seperti CSRT, KCF, MOSSE, atau MIL. Tracking dilakukan dengan memperbarui posisi objek secara terus-menerus sehingga objek tetap diberi kotak penanda selama masih terdeteksi. Hasilnya menunjukkan bahwa setiap tracker memiliki karakteristik berbeda, di mana CSRT lebih akurat namun lebih lambat, KCF lebih cepat tetapi sedikit kurang stabil, dan MOSSE sangat cepat namun kurang presisi saat objek bergerak cepat atau berubah arah.

## Video Stabilization (Opsional / Pengayaan)

Video stabilization adalah teknik untuk mengurangi  getaran dalam video, menghasilkan footage yang lebih halus dan stabil untuk pengalaman menonoton yang lebih baik.

### Pengenalan singkat konsep stabilisasi video (feature matching & warping).

**Prinsip Kerja Video Stabilization:**

#### **1. Feature Detection & Matching**
- **Deteksi keypoints**: Menggunakan `cv2.goodFeaturesToTrack()` untuk menemukan titik-titik karakteristik yang mudah dikenali
- **Feature matching**: Mencocokkan keypoints yang sama antar frame berurutan
- **Tracking motion**: Menggunakan `cv2.calcOpticalFlowPyrLK()` untuk melacak pergerakan fitur-fitur tersebut

#### **2. Motion Estimation**
- **Transformasi geometris**: Menghitung bagaimana frame bergerak relatif terhadap frame sebelumnya
- **Parameter motion**: Translation (x,y), rotation, dan scaling menggunakan `cv2.estimateAffinePartial2D()`
- **Motion vector**: Mendapatkan vektor perpindahan untuk setiap frame

#### **3. Trajectory Smoothing**
- **Raw trajectory**: Kumpulan semua motion vector (biasanya bergelombang/noisy)
- **Smooth trajectory**: Aplikasi moving average atau filter lain untuk menghaluskan pergerakan
- **Stabilization**: Selisih antara raw dan smooth trajectory menjadi koreksi yang dibutuhkan

#### **4. Frame Correction & Warping**
- **Geometric transformation**: Menerapkan koreksi menggunakan `cv2.warpAffine()` atau `cv2.warpPerspective()`
- **Frame warping**: Merotasi, mentranslasi, atau menskalakan frame untuk mengkompensasi goyangan
- **Border handling**: Mengatasi area kosong akibat transformasi (crop, interpolation, atau border replication)

**Kelebihan & Keterbatasan:**
- ✅ **Efektif**: Mengurangi goyangan secara signifikan untuk kebanyakan kasus
- ✅ **Non-destructive**: Tidak merusak kualitas visual secara drastis
- ❌ **Cropping**: Memerlukan sedikit crop di tepi frame untuk ruang koreksi
- ❌ **Rolling shutter**: Kurang efektif untuk distorsi rolling shutter kamera smartphone

In [5]:
# Video Stabilization versi lightweight real-time (beda konsep dari dosen)
import cv2
import numpy as np
import os

PATH_VIDEO = os.path.join(os.getcwd(), 'data', 'unstabilized_video.mp4')

cap = cv2.VideoCapture(PATH_VIDEO)

if not cap.isOpened():
    print("Video tidak bisa dibuka")
    exit()

ret, prev = cap.read()
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

# ambil fitur awal
prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=80, qualityLevel=0.3, minDistance=7)

# 🔥 bedanya: pakai velocity smoothing (bukan trajectory)
vx, vy = 0, 0
beta = 0.85

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None)

    if next_pts is not None:
        good_prev = prev_pts[status == 1]
        good_next = next_pts[status == 1]

        if len(good_prev) > 0:
            dx = np.median(good_next[:,0] - good_prev[:,0])
            dy = np.median(good_next[:,1] - good_prev[:,1])

            # 🔥 smoothing beda (velocity-based EMA)
            vx = beta * vx + (1 - beta) * dx
            vy = beta * vy + (1 - beta) * dy

            M = np.array([[1, 0, -vx],
                          [0, 1, -vy]], dtype=np.float32)

            stabilized = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))
        else:
            stabilized = frame

    else:
        stabilized = frame

    prev_gray = gray
    prev_pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=80, qualityLevel=0.3, minDistance=7)

    cv2.imshow("KAMU - EMA Stabilization (DIFFERENT)", stabilized)

    # stop pakai y (biar beda)
    if cv2.waitKey(1) & 0xFF == ord('y'):
        break

cap.release()
cv2.destroyAllWindows()

>>Pada percobaan ini saya melakukan video stabilization menggunakan pendekatan yang lebih sederhana dan real-time dibandingkan metode dosen. Saya tidak menggunakan perhitungan trajectory penuh, melainkan langsung menghitung pergeseran antar frame menggunakan optical flow. Nilai pergeseran tersebut kemudian saya haluskan menggunakan exponential moving average berbasis velocity sehingga perubahan pergerakan menjadi lebih stabil secara bertahap. Hasil smoothing ini langsung diterapkan pada setiap frame menggunakan transformasi affine, sehingga proses stabilisasi dapat berjalan lebih ringan tanpa perlu menyimpan seluruh data transformasi terlebih dahulu.